<a href="https://colab.research.google.com/github/dang710206/Baitap-AI-/blob/main/Bt12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#BT12
import folium
import numpy as np
from itertools import permutations
import random

# ================== 1. Dữ liệu: Tọa độ TP.HCM ==================
# Depot (Kho)
depots = {
    'Kho1': [10.7769, 106.7009],   # Quận 1
    'Kho2': [10.8122, 106.7180]    # Bình Thạnh
}

# 10 điểm giao hàng
delivery_points = {
    'P1': [10.7828, 106.6950],   # Quận 3
    'P2': [10.7295, 106.7218],   # Quận 7
    'P3': [10.8231, 106.6297],   # Gò Vấp
    'P4': [10.8012, 106.7114],   # Thủ Đức
    'P5': [10.7962, 106.6635],   # Tân Bình
    'P6': [10.7500, 106.6500],   # Bình Thạnh (gần kho2)
    'P7': [10.7700, 106.6800],   # Quận 1
    'P8': [10.7900, 106.7100],   # Phú Nhuận
    'P9': [10.7400, 106.6700],   # Quận 5
    'P10': [10.8200, 106.6400]   # Tân Phú
}

all_points = {**depots, **delivery_points}
point_names = list(all_points.keys())
coords = np.array([all_points[name] for name in point_names])

# Tính ma trận khoảng cách (km) - khoảng cách Euclidean x100 cho đơn giản
def haversine_distance(coord1, coord2):
    R = 6371  # Bán kính Trái Đất (km)
    lat1, lon1 = np.radians(coord1)
    lat2, lon2 = np.radians(coord2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

dist_matrix = np.zeros((len(coords), len(coords)))
for i in range(len(coords)):
    for j in range(len(coords)):
        dist_matrix[i, j] = haversine_distance(coords[i], coords[j])

# Gán index cho depot và points
depot_idx = [0, 1]  # Kho1=0, Kho2=1
customer_idx = list(range(2, len(point_names)))

# ================== 2. Heuristic: Nearest Neighbor + phân bổ kho ==================
def nearest_neighbor_route(start_idx, customers, capacity=100):
    route = [start_idx]
    remaining = customers[:]
    current = start_idx
    total_dist = 0
    load = 0

    while remaining and load < capacity:
        distances = [(i, dist_matrix[current, i]) for i in remaining]
        next_point = min(distances, key=lambda x: x[1])[0]
        route.append(next_point)
        total_dist += dist_matrix[current, next_point]
        load += 10  # giả sử mỗi điểm 10 đơn
        current = next_point
        remaining.remove(next_point)

    # Trở về kho gần nhất
    return_to = min(depot_idx, key=lambda d: dist_matrix[current, d])
    route.append(return_to)
    total_dist += dist_matrix[current, return_to]

    return route, total_dist

# Phân bổ khách cho từng kho (gần nhất)
def assign_to_depots(customers):
    assignments = {0: [], 1: []}
    for cust in customers:
        dist_to_depots = [dist_matrix[cust, d] for d in depot_idx]
        closest_depot = np.argmin(dist_to_depots)
        assignments[closest_depot].append(cust)
    return assignments

# Tìm tuyến cho tất cả xe
assignments = assign_to_depots(customer_idx)
routes = []
total_distance_optimized = 0

for depot_id, cust_list in assignments.items():
    if cust_list:
        route, dist = nearest_neighbor_route(depot_id, cust_list)
        routes.append((depot_id, route, dist))
        total_distance_optimized += dist

print("=== Kết quả tối ưu (Heuristic) ===")
for i, (depot_id, route, dist) in enumerate(routes):
    print(f"Tuyến xe {i+1} từ {point_names[depot_id]}: ")
    print(" -> ".join([point_names[idx] for idx in route]))
    print(f"Khoảng cách: {dist:.2f} km\n")

print(f"Tổng khoảng cách tối ưu: {total_distance_optimized:.2f} km")

# ================== 3. Phương án không tối ưu (ngẫu nhiên) ==================
random_routes = []
total_distance_random = 0
remaining = customer_idx[:]
random.shuffle(remaining)

for i in range(3):  # 3 xe
    if not remaining:
        break
    start = random.choice(depot_idx)
    route = [start]
    current = start
    for _ in range(min(4, len(remaining))):  # mỗi xe max 4 điểm
        if not remaining:
            break
        next_p = remaining.pop(0)
        route.append(next_p)
        total_distance_random += dist_matrix[current, next_p]
        current = next_p
    # Trở về kho
    return_to = random.choice(depot_idx)
    route.append(return_to)
    total_distance_random += dist_matrix[current, return_to]
    random_routes.append(route)

print(f"Tổng khoảng cách phương án ngẫu nhiên: {total_distance_random:.2f} km")
print(f"Tiết kiệm được: {total_distance_random - total_distance_optimized:.2f} km ({((total_distance_random - total_distance_optimized)/total_distance_random*100):.1f}%)")

# ================== 4. Vẽ bản đồ ==================
m = folium.Map(location=[10.7769, 106.7009], zoom_start=12, tiles="cartodb positron")

colors = ['red', 'blue', 'green', 'purple']

# Vẽ các điểm
for name, coord in all_points.items():
    if name in depots:
        folium.Marker(coord, popup=name, icon=folium.Icon(color='black', icon='warehouse')).add_to(m)
    else:
        folium.Marker(coord, popup=name, icon=folium.Icon(color='orange')).add_to(m)

# Vẽ tuyến đường tối ưu
for i, (depot_id, route, _) in enumerate(routes):
    route_coords = [all_points[point_names[idx]] for idx in route]
    folium.PolyLine(route_coords, color=colors[i % len(colors)], weight=5, opacity=0.8,
                    popup=f"Tuyến xe {i+1} (Tối ưu)").add_to(m)

# Lưu bản đồ
m.save("ban_do_toi_uu_tuyen_giao_hang_23_12.html")
print("✅ Bản đồ đã được tạo: ban_do_toi_uu_tuyen_giao_hang_23_12.html")

m

=== Kết quả tối ưu (Heuristic) ===
Tuyến xe 1 từ Kho1: 
Kho1 -> P1 -> P8 -> P7 -> P5 -> P10 -> P3 -> P6 -> P9 -> P2 -> Kho1
Khoảng cách: 37.39 km

Tuyến xe 2 từ Kho2: 
Kho2 -> P4 -> Kho2
Khoảng cách: 2.84 km

Tổng khoảng cách tối ưu: 40.23 km
Tổng khoảng cách phương án ngẫu nhiên: 79.09 km
Tiết kiệm được: 38.86 km (49.1%)
✅ Bản đồ đã được tạo: ban_do_toi_uu_tuyen_giao_hang_23_12.html
